I used this file to ingest, parse and store information from Wattbot2025/metadata.csv in a format that could be used to create a RAG chatbot. These files and folders are stored under extracted_data.

In [2]:
import pandas as pd
from mistralai import Mistral
from dotenv import load_dotenv
import datauri
import os
import re
import json
import base64
from openai import OpenAI

In [3]:
#Reading in a list of URLs
metadata = pd.read_csv("WattBot2025/metadata.csv",encoding='latin-1')
urlList = metadata["url"].tolist()

In [13]:
#Creating a folder for each document which contains a markdown of the PDF and corresponding images as .jpegs
load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

for url in urlList:
    ocr_response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "docume4nt_url",
            "document_url": url,
        },
        include_image_base64=True,
    )

    doc_index = urlList.index(url) + 1
    doc_dir = os.path.join("extracted_data", f"document_{doc_index}")
    os.makedirs(doc_dir, exist_ok=True)  # This prevents the FileNotFoundError

    with open(os.path.join(doc_dir,f"doc_{urlList.index(url)+1}.md"),"wt") as f:
        for page in ocr_response.pages:
            f.write(page.markdown)
            for image in page.images:
                im = datauri.parse(image.image_base64)
                with open(os.path.join(doc_dir,image.id),"wb") as fi:
                    fi.write(im.data)

KeyboardInterrupt: 

In [4]:
#Creating a folder for the last document which contains a markdown of the PDF and corresponding images as .jpegs
doc_index = 32
doc_dir = os.path.join("extracted_data", f"document_{doc_index}")
os.makedirs(doc_dir, exist_ok=True)

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": "https://arxiv.org/pdf/2508.14170",
    },
    include_image_base64=True,
)
with open(os.path.join(doc_dir,f"doc_{32}.md"),"wt") as f:
        for page in ocr_response.pages:
            f.write(page.markdown)
            for image in page.images:
                im = datauri.parse(image.image_base64)
                with open(os.path.join(doc_dir,image.id),"wb") as fi:
                    fi.write(im.data)    

NameError: name 'client' is not defined

In [4]:
#Additional data collection from metadata.csv (this will help us assign relevant important information to .json files)
metadata = pd.read_csv("Wattbot2025/metadata.csv",encoding='latin-1')
ref_id_list = metadata["id"].tolist()
ref_url_list = metadata["url"].tolist()

In [21]:
load_dotenv()
openai_api_key = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=openai_api_key)
def encode_image(img_path):
    with open(img_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def image_description(img_path):
    try:
        image = encode_image(img_path)
        prompt = f"""
        You are a perceptive and descriptive analyst who can understand the details
        of a technical image or graph and offer a succinct description of it. 
        Follow the steps below to create your description:
        1. I will give you an image as input.
        2. Based on the image input, generate a short, succinct yet information heavy description
        of the image no longer than 3 sentences. The sentence should be such that it could be
        used by an AI program to understand what an image is displaying.
        """
        response = client.responses.create(
            model="gpt-4o",
            input=[
                {
                    "role": "user",
                    "content": [
                        { "type": "input_text", "text": f"Here is the prompt {prompt}" },
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{image}",
                        }
                    ]
                }
            ]
        )
        return response.output_text
    except Exception as e:
        return "No Image Description"
    
context = "erstand the opportunity for improving MoE layer performance, we also perform a kernel-level analysis within the MoE layer. Fig. 7 illustrates the architecture of the MoE layer in both Mixtral and BlackMamba models. Each expert in BlackMamba consists of a standard Feed-Forward Network (FFN) layer with two serially connected weight matrices (W1 and W2) and a Gelu activation layer between. In contrast, experts in Mixtral are FFN layers with Swish-Gated Linear Units, involving an additional weight\n\n![img-6.jpeg](img-6.jpeg)\n\n![img-7.jpeg](img-7.jpeg)\nFig. 7. Expert architectures for Mixtral (top) and BlackMamba (bottom).\n\nmatrix (W3) in parallel with W1.\n\nFig. 6 shows the kernel-level MoE time breakdown. The figure clearly shows that matrix multiplication (W1, W2, and W3) is the largest component of the MoE layer for both BlackMamba and Mixtral. As batch size and sparsity increase, so does computational demand, prolonging matrix multiplication latency. The de-quantization operation in Mixtral fine-tuning also beco"
print(image_description("extracted_data/document_31/img-6.jpeg"))

The image is a quadrant scatter plot comparing the energy usage (in Wh) and accuracy of different language models across four datasets: news, yelp, tomatoes, and emotion. Each plot separates models into categories (Deepeek, Other LLM, Traditional), with key models such as DS Qwen, DS Llama, and Linear Embedding highlighted for their respective energy-accuracy balance. The plots demonstrate that more complex models tend to have higher energy consumption across all datasets, with linear embedding consistently showing lower energy use.


In [24]:
#For each document, creates a metadata .json file with identifying information for each image like relevant context and location. 
for folder_name in os.listdir("extracted_data"):
    if folder_name.startswith("document_") and folder_name!="document_1":
        all_images = []
        doc_num = folder_name.split("_")[-1]
        md_path = f"extracted_data/{folder_name}/doc_{doc_num}.md"
        ref_id = ref_id_list[int(doc_num)-1]
        ref_url = ref_url_list[int(doc_num)-1]
        with open(md_path,"r",encoding="utf-8") as f:
            content = f.read()
            matches = re.finditer(r"!\[(.*?)\]\((.*?)\)", content)
            for match in matches:
                img = match[1]
                img_path = f"extracted_data/{folder_name}/{img}"
                start,end = match.span()
                neighborhood = content[max(0, start - 500) : min(len(content), end + 500)]
                img_description = image_description(img_path)
                image_data = {
                    "doc_path" : md_path,
                    "ref_id": ref_id,
                    "ref_url": ref_url,
                    "img_id": img,
                    "img_uid": ref_id + img,
                    "context": neighborhood,
                    "img_path": img_path,
                    "img_description": img_description
                }
                all_images.append(image_data)
        metadata_path = f"extracted_data/{folder_name}/metadata_{doc_num}.json"
        with open(metadata_path,"w",encoding="utf-8") as m:
            json.dump(all_images,m,indent=4)

In [73]:
#Creating a .json folder to match each document folder to the ref_id in metadata.csv
metadata = pd.read_csv("Wattbot2025/metadata.csv",encoding='latin-1')
idList = metadata["id"]

meta = pd.DataFrame(columns=["ref_id","ref_url","document_path","markdown"])
for url in urlList:
    index = urlList.index(url)
    new_doc = {"ref_id":idList[index], "ref_url": url, "document_path":f"extracted_data/document_{index+1}", "markdown":f"extracted_data/document_{index+1}/doc_{index+1}.md"}
    meta.loc[len(meta)] = new_doc

meta.to_csv("meta.csv")

In [7]:
#Creating a file for each document that stores information about its id and url. 
for folder_name in os.listdir("extracted_data"):
    if folder_name.startswith("document_"):
        doc_num = folder_name.split("_")[-1]
        ref_id = ref_id_list[int(doc_num)-1]
        ref_url = ref_url_list[int(doc_num)-1]
        doc_data = {
            "ref_id": ref_id,
            "ref_url": ref_url
        }
        doc_data_path = f"extracted_data/{folder_name}/doc_data_{doc_num}.json"
        with open(doc_data_path,"w",encoding="utf-8") as m:
            json.dump(doc_data,m,indent=4)